In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the project's src/ package importable regardless of Jupyter's cwd.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features import (
    load_aapl_intraday,
    add_candle_features,
    add_return_features,
    add_volatility_volume_features,
    add_technical_indicators,
    add_target,
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    RAW_SNAPSHOT_PATH,
    PROCESSED_TRAINING_PATH,
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load the shared 5-minute intraday snapshot via src/features.py -- the same
# source file used in notebooks/exploration.ipynb, already quality-checked there.
df = load_aapl_intraday(RAW_SNAPSHOT_PATH)

print(f"AAPL data shape: {df.shape}")
print(f"Period: {df.index.min()} to {df.index.max()}")
df.head()

AAPL data shape: (390, 5)
Period: 2026-09-03 09:30:00-04:00 to 2026-09-10 15:55:00-04:00


Price,Close,High,Low,Open,Volume
Datetime,,,,,
2026-09-03 09:30:00-04:00,325.619995,327.500000,324.250000,324.947693,1747712
2026-09-03 09:35:00-04:00,326.450012,326.471008,324.109985,325.540009,543838
2026-09-03 09:40:00-04:00,326.484985,326.670013,325.400513,326.450012,415504
2026-09-03 09:45:00-04:00,326.589996,326.970001,326.190002,326.470001,407058
2026-09-03 09:50:00-04:00,326.269989,326.919891,326.000000,326.559998,402373


# Feature Engineering — AAPL

Builds a per-5-minute-bar feature and target table for AAPL. All the actual
feature/target logic now lives in `src/features.py`; this notebook just calls
it step by step so each stage's output can be inspected. As noted in
`exploration.ipynb`: chronological splitting and evaluation belong in
`src/train.py`, and future prices must never enter model inputs.

1. Candle-shape features (range, body)
2. Lagged return features (5m / 15m / 30m), gap-masked
3. Rolling volatility and relative volume, grouped by trading day
4. Technical indicators (RSI, MACD, Bollinger Band width, ATR)
5. Target: 30-minute-ahead direction and return, gap-masked
6. Assemble, drop warm-up/tail NaNs, and persist to `data/processed/aapl_training.parquet`

In [3]:
# --- Candle-shape features (src/features.py: add_candle_features) ---
df = add_candle_features(df)
df[["range_pct", "body_pct"]].describe()

Price,range_pct,body_pct
count,390.000000,390.000000
mean,0.238099,0.003039
std,0.187155,0.178155
min,0.041108,-0.746005
25%,0.122856,-0.077090
50%,0.181753,0.002375
75%,0.283398,0.075922
max,1.343108,0.823885


In [4]:
# --- Lagged return features, gap-masked (src/features.py: add_return_features) ---
df = add_return_features(df)
df[["return_5m_pct", "return_15m_pct", "return_30m_pct"]].describe()

Price,return_5m_pct,return_15m_pct,return_30m_pct
count,385.000000,375.000000,360.000000
mean,0.001378,0.003153,0.010435
std,0.173347,0.317930,0.442277
min,-0.753509,-1.203085,-1.526207
25%,-0.074987,-0.136740,-0.204117
50%,0.003172,0.015054,0.022563
75%,0.074935,0.129944,0.198658
max,0.808611,1.849810,1.823214


In [5]:
# --- Rolling volatility and relative volume (src/features.py: add_volatility_volume_features) ---
df = add_volatility_volume_features(df)
df[["volatility_30m", "volume_relative"]].describe()

Price,volatility_30m,volume_relative
count,360.000000,295.000000
mean,0.138270,1.039000
std,0.099552,0.660769
min,0.026647,0.276556
25%,0.068653,0.679898
50%,0.106479,0.853491
75%,0.174633,1.170848
max,0.535233,6.075438


In [6]:
# --- Technical indicators (src/features.py: add_technical_indicators) ---
df = add_technical_indicators(df)
df[["rsi_14", "macd_diff", "bb_width_pct", "atr_pct"]].describe()

Price,rsi_14,macd_diff,bb_width_pct,atr_pct
count,377.000000,357.000000,371.000000,390.000000
mean,47.924895,-0.001874,1.210827,0.232641
std,13.546157,0.220269,0.881279,0.111880
min,14.305293,-0.572875,0.137227,0.000000
25%,39.037670,-0.115154,0.409970,0.147433
50%,48.832975,0.028722,0.936018,0.212281
75%,58.504649,0.125214,1.964523,0.285978
max,74.513995,0.673299,3.692283,0.544953


In [7]:
# --- Target: 30-minute-ahead direction and return, gap-masked (src/features.py: add_target) ---
df = add_target(df)
df[["target_return_30m_pct", "target_up_30m", "target_time"]].tail(10)

Price,target_return_30m_pct,target_up_30m,target_time
Datetime,,,
2026-09-10 15:10:00-04:00,-0.122801,0,2026-09-10 15:40:00-04:00
2026-09-10 15:15:00-04:00,0.045133,1,2026-09-10 15:45:00-04:00
2026-09-10 15:20:00-04:00,0.004602,1,2026-09-10 15:50:00-04:00
2026-09-10 15:25:00-04:00,0.402723,1,2026-09-10 15:55:00-04:00
2026-09-10 15:30:00-04:00,NaN,<NA>,NaT
2026-09-10 15:35:00-04:00,NaN,<NA>,NaT
2026-09-10 15:40:00-04:00,NaN,<NA>,NaT
2026-09-10 15:45:00-04:00,NaN,<NA>,NaT
2026-09-10 15:50:00-04:00,NaN,<NA>,NaT


In [8]:
# --- Assemble the final training table ---
df_features = df[FEATURE_COLUMNS + TARGET_COLUMNS].copy()

# Drop rows with missing values. NaNs come from two sources only:
#   1. Indicator/rolling warm-up at the start of the series (not enough history yet).
#   2. The last few rows, which have no future bar to compute a target from.
# The DatetimeIndex keeps everything in chronological order throughout, so this
# never shuffles data and features are never computed from future information.
rows_before = len(df_features)
df_features = df_features.dropna()
print(f"Dropped {rows_before - len(df_features)} rows with missing feature/target values")
print(f"Final shape: {df_features.shape}")

df_features.head()

Dropped 139 rows with missing feature/target values
Final shape: (251, 14)


Price,range_pct,body_pct,return_5m_pct,return_15m_pct,return_30m_pct,volatility_30m,volume_relative,rsi_14,macd_diff,bb_width_pct,atr_pct,target_return_30m_pct,target_up_30m,target_time
Datetime,,,,,,,,,,,,,,
2026-09-03 12:15:00-04:00,0.148644,-0.118166,-0.115201,-0.084869,-0.027294,0.103248,0.537605,56.389076,-0.126018,0.806907,0.244817,-0.066740,0,2026-09-03 12:45:00-04:00
2026-09-03 12:20:00-04:00,0.317817,0.222969,0.226003,0.247293,0.256326,0.134331,1.256358,62.484526,-0.088354,0.763741,0.249519,-0.323867,0,2026-09-03 12:50:00-04:00
2026-09-03 12:25:00-04:00,0.173839,-0.095381,-0.095381,0.015054,0.090941,0.144248,0.858727,58.744366,-0.086333,0.743166,0.244335,-0.342289,0,2026-09-03 12:55:00-04:00
2026-09-03 12:30:00-04:00,0.112082,0.021203,0.012149,0.142571,0.057581,0.143489,0.694106,59.080027,-0.083892,0.613292,0.234860,-0.430154,0,2026-09-03 13:00:00-04:00
2026-09-03 12:35:00-04:00,0.122764,-0.065128,-0.063613,-0.146803,0.100126,0.137561,0.736669,56.488142,-0.097085,0.358809,0.226992,-0.421335,0,2026-09-03 13:05:00-04:00


In [9]:
# Persist the training table for src/train.py to consume.
# Saved as Parquet (not CSV) to preserve dtypes and the tz-aware DatetimeIndex.
PROCESSED_TRAINING_PATH.parent.mkdir(parents=True, exist_ok=True)
df_features.to_parquet(PROCESSED_TRAINING_PATH)

print(f"Saved {df_features.shape[0]} rows x {df_features.shape[1]} columns to {PROCESSED_TRAINING_PATH}")

Saved 251 rows x 14 columns to D:\Projects\Alpha-Predictor\data\processed\aapl_training.parquet


## 6. Sanity Checks

In [10]:
# Class balance of the target -- important to know before training, since a
# heavily imbalanced target (e.g. 90% "up") would make plain accuracy misleading.
df_features["target_up_30m"].value_counts(normalize=True).rename("share")

target_up_30m
1    0.589641
0    0.410359
Name: share, dtype: Float64